In [ ]:
import json
import re
from copy import deepcopy

# Filenames
DATASET_IN = "data4.jsonld"
DATASET_OUT = "data5.jsonld"
THESAURUS_MAP = "thesaurusTerms.json"

# Properties that should link to thesaurus
THESAURUS_PROPERTIES = {
    "write:hasGraffitiType",
    "write:hasGraffitiStyle",
    "write:hasGraffitiGenre",
    "wdt:P9302",
    "write:hasChineseConcept",
    "write:hasChineseVisualElement"
}

# Old dataset labels -> new thesaurus labels
RENAMED_TERMS = {
    "Birds and worms script": "bird and insect script",
    "Cursive-running script": "cursive running script",
    "Horizontal scroll": "handscroll",
    "Mist": "cloud and mist",
    "Puppet": "puppet style",
    "Three distances method": "three distances",
    "Turbulent ocean waves": "turbulent ocean waves motif",
    "White and blue porcelains": "blue and white porcelain",
    "Zen/Chan": "chan",
    "Daoist talismanic script": "talismanic script",
    "Chinese literati/gentleman (junzi)": "junzi"
    
}

# Normalize labels for matching
def normalize_label(label: str) -> str:
    label = label.lower()
    label = re.sub(r"\s*\([^)]*\)", "", label)
    label = re.sub(r"\s*[/\-–]\s*", "/", label)
    label = re.sub(r"\s+", " ", label)
    return label.strip()

# Build normalized alias map
RENAMED_TERMS_NORMALIZED = {
    normalize_label(old): normalize_label(new)
    for old, new in RENAMED_TERMS.items()
}

# Load thesaurus map
with open(THESAURUS_MAP, "r", encoding="utf-8") as f:
    thesaurus_map = json.load(f)

# Build label lookup from thesaurus
label_to_entry = {}
for entry in thesaurus_map.values():
    lab = entry.get("rdfs:label")
    if lab:
        label_to_entry[normalize_label(lab)] = entry

# Set to store unmatched thesaurus candidates
unmatched_terms = set()

def convert_list_entry(key, entry):
    """
    Convert an entry dict:
    - Only attempt thesaurus match for THESAURUS_PROPERTIES
    - Replace with thesaurus object if exact match
    - Also resolve known renamed terms
    - Track unmatched if could have matched
    """
    e = deepcopy(entry)

    if "@id" in e:
        e.pop("@language", None)
        return e

    if "@value" in e and isinstance(e["@value"], str):
        if key in THESAURUS_PROPERTIES:
            original_value = e["@value"]
            norm = normalize_label(original_value)

            # First, rewrite old labels to the new thesaurus labels
            norm = RENAMED_TERMS_NORMALIZED.get(norm, norm)

            matched = label_to_entry.get(norm)
            if matched:
                new = {
                    "@id": matched["@id"],
                    "rdfs:label": matched["rdfs:label"]
                }
                if "property_label" in e:
                    new["property_label"] = e["property_label"]
                return new
            else:
                unmatched_terms.add(original_value)
                return e
        else:
            return e

    # If @value is a dict, unwrap if it contains @id
    if "@value" in e and isinstance(e["@value"], dict):
        nested = e["@value"]
        if "@id" in nested:
            new = {"@id": nested["@id"]}
            if "rdfs:label" in nested:
                new["rdfs:label"] = nested["rdfs:label"]
            elif "skos:prefLabel" in nested:
                new["rdfs:label"] = nested["skos:prefLabel"]
            if "property_label" in e:
                new["property_label"] = e["property_label"]
            return new
        e.pop("@language", None)
        return e

    # URI/resource items
    if e.get("type") in ("uri", "resource:item") and "@id" in e:
        new = {"@id": e["@id"]}
        if "rdfs:label" in e:
            new["rdfs:label"] = e["rdfs:label"]
        if "property_label" in e:
            new["property_label"] = e["property_label"]
        return new

    e.pop("@language", None)
    return e

def traverse_and_convert(obj):
    if isinstance(obj, dict):
        new = {}
        for k, v in obj.items():
            if isinstance(v, list):
                new_list = []
                for item in v:
                    if isinstance(item, dict):
                        converted = convert_list_entry(k, item)
                        new_list.append(converted)
                    else:
                        new_list.append(item)
                new[k] = new_list
            else:
                new[k] = traverse_and_convert(v)
        return new
    elif isinstance(obj, list):
        return [traverse_and_convert(x) for x in obj]
    else:
        return obj

# Load dataset
with open(DATASET_IN, "r", encoding="utf-8") as f:
    dataset = json.load(f)

# Apply conversion
converted = traverse_and_convert(dataset)

# Save converted dataset
with open(DATASET_OUT, "w", encoding="utf-8") as f:
    json.dump(converted, f, indent=2, ensure_ascii=False)

# Save unmatched thesaurus candidates
with open("unmatched_terms.json", "w", encoding="utf-8") as f:
    json.dump(sorted(unmatched_terms), f, indent=2, ensure_ascii=False)

print(f"✅ Done — written {DATASET_OUT}")
print(f"⚠️ Unmatched thesaurus terms saved to unmatched_terms.json ({len(unmatched_terms)} terms)")

✅ Done — written data5.jsonld
⚠️ Unmatched thesaurus terms saved to unmatched_terms.json (0 terms)
